# Machine Learning Report
## Hotel Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  
**Project:** Hotel Occupancy Analysis and Prediction

---

## Purpose of the ML Stage

The ML stage tests whether lagged Google Trends features improve daily hotel occupancy prediction beyond hotel identity, seasonality, and recent occupancy history.

**Main takeaway:** Lagged Google Trends improves the learned models, but NaivePersistence remains the strongest overall benchmark. Therefore, Google Trends is useful as a supporting signal, while daily occupancy is mainly driven by recent occupancy and seasonality.

## Modeling Strategy Used So Far

Two learned settings were compared:

1. **Baseline model:** hotel identity, calendar/seasonality features, and past occupancy lags.
2. **Baseline + Trends model:** the same baseline features plus selected lagged Google Trends variables.

Simple benchmark rules were also added later to test whether the learned models actually outperform basic time-series predictions.

## Feature Engineering

The model uses calendar features, cyclical seasonality features, hotel-specific occupancy lags, and selected lagged Google Trends variables. Occupancy lags were created separately for each hotel, while Google Trends lags were created on a unique date-level table before being merged back into the hotel-date dataset.

## Models Compared

Two learned models were used: **Ridge Regression** as a regularized linear benchmark and **Random Forest Regressor** as a non-linear model that can capture interactions between seasonality, hotel identity, lagged occupancy, and Google Trends.

## Validation Design

The evaluation started with a chronological train-test split, then became stricter through fair same-window validation and walk-forward validation. Finally, NaivePersistence and SeasonalNaive7 were added as simple benchmark rules.

## Current First-Pass ML Findings

After correcting the Google Trends lag alignment, the first-pass result remained the same: adding lagged Google Trends improved the learned model.

- Best baseline-only RMSE: **approximately 5.87**
- Best baseline + Trends RMSE: **approximately 4.80**

This supports the idea that lagged Google Trends adds useful signal, but it is not the final conclusion because later benchmark analysis shows that NaivePersistence is stronger.

## Hotel-Level Prediction Plots

These figures show actual vs predicted occupancy for the best non-robustness trends-augmented learned model.

### Azura Deluxe
![Azura Deluxe prediction](Figures/actual_vs_pred_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel prediction](Figures/actual_vs_pred_side_mare_hotel.png)

## Hotel-wise Normalization Robustness in the ML Stage

The EDA stage already tested whether pooled correlation findings survive hotel-wise normalization. The ML stage extends that idea by asking whether the incremental value of lagged Google Trends remains visible when the target is normalized within each hotel. This matters because the two hotels do not operate at exactly the same occupancy level or variability scale.

The normalized target was `target_hotel_z = (occupancy_rate - train_hotel_mean) / train_hotel_std`. Hotel mean and standard deviation were computed from the **train period only**, then applied to both train and test, which keeps the normalization leakage-safe.

- Best baseline-only RMSE after back-transformation: **approximately 6.19**
- Best baseline + Trends RMSE after back-transformation: **approximately 5.67**

This suggests that the value of lagged Google Trends is **not only an artifact of pooling two hotels with different average occupancy levels**. Still, this is a robustness check within the learned-model framework. It does not by itself show that the learned models outperform simple persistence-style rules.

## Hotel-normalized Prediction Plots

These figures show actual vs predicted occupancy for the best hotel-normalized robustness model, back-transformed to raw occupancy scale.

### Azura Deluxe
![Azura Deluxe normalized-target robustness prediction](Figures/actual_vs_pred_hotel_z_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel normalized-target robustness prediction](Figures/actual_vs_pred_hotel_z_side_mare_hotel.png)

## Fair Same-Window Validation

The first-pass comparison was useful, but it still allowed the baseline and baseline + Trends models to be built on slightly different non-missing datasets. The fair same-window pass fixes this by forcing both setups to use the **same rows** and the **same test period**.

- best baseline-only RMSE: **4.974**
- best baseline + Trends RMSE: **4.798**

The best learned model again remained **Random Forest**, and the Trends-augmented version still performed better than the learned baseline-only version. This matters because it shows that the earlier Trends gain does **not disappear** when the comparison is made on a stricter, methodologically cleaner evaluation window.

### Fair same-window prediction plots

#### Azura Deluxe
![Azura Deluxe fair same-window prediction](Figures/actual_vs_pred_same_window_azura_deluxe.png)

#### Side Mare Hotel
![Side Mare Hotel fair same-window prediction](Figures/actual_vs_pred_same_window_side_mare_hotel.png)

## Walk-Forward Validation

The walk-forward stage is a stronger time-series validation step than a single holdout. Instead of testing on only one final block, it evaluates the same learned-model logic across several future periods.

- best baseline-only mean RMSE across folds: **8.166**
- best baseline + Trends mean RMSE across folds: **8.035**

So the average improvement from Trends remained **positive but modest**. The walk-forward learned-model results are more variable than the single holdout results. Some folds are much harder than others, and overall performance is less stable across time. This suggests that seasonality and timing effects remain dominant, the Google Trends contribution is useful but not large, and final claims should rely more on repeated validation than on a single test split.

### Walk-forward validation figures

#### RMSE by fold
![Walk-forward RMSE by fold](Figures/walk_forward_rmse_by_fold.png)

#### Mean RMSE summary
![Walk-forward mean RMSE summary](Figures/walk_forward_mean_rmse_summary.png)

## Naive Benchmark Comparison

At this point, the key remaining question was not whether lagged Google Trends helped the learned models relative to each other. That had already been shown. The more important question became whether the learned models beat simple time-series benchmark rules.

- **NaivePersistence RMSE: 4.068**
- **SeasonalNaive7 RMSE: 7.808**
- best learned model (`baseline_plus_trends / RandomForest`) RMSE: **4.798**

This means that while the Trends-augmented Random Forest improved on the learned baseline, it still **did not beat NaivePersistence**.

Across folds:

- **NaivePersistence mean RMSE: 4.170**
- **SeasonalNaive7 mean RMSE: 8.350**
- best learned model (`baseline_plus_trends / RandomForest`) mean RMSE: **8.035**

Again, NaivePersistence remained the strongest benchmark overall. The learned models, including the Trends-augmented ones, did not surpass it.

### Naive benchmark figures

#### Fair same-window RMSE with naive benchmarks
![Fair same-window with naive benchmarks](Naive_Benchmark/same_window_rmse_with_naive_benchmarks.png)

#### Walk-forward RMSE with naive benchmarks
![Walk-forward RMSE with naive benchmarks](Naive_Benchmark/walk_forward_rmse_with_naive_benchmarks.png)

#### Walk-forward mean RMSE with naive benchmarks
![Walk-forward mean RMSE with naive benchmarks](Naive_Benchmark/walk_forward_mean_rmse_with_naive_benchmarks.png)

## Current Limitations of the ML Stage

Although the ML stage is now much stronger than the initial first pass, it is still not the final modeling design. The main limitation is that the learned models do not beat **NaivePersistence**. The current learned pipeline improves over the learned baseline, but it still cannot beat the strongest simple benchmark. Walk-forward performance is also unstable, meaning predictive difficulty changes across time periods. The current Trends feature set remains compact, Google Trends is only one external signal family, and daily occupancy may simply be very persistence-dominated.

## Interpretation

The ML evidence now supports a careful conclusion. **Seasonality and recent occupancy remain the dominant drivers**, while **lagged Google Trends adds incremental predictive value** inside the learned-model framework. This signal survives the first-pass, robustness, fair same-window, and walk-forward learned-model comparisons. However, the current learned models still **do not outperform NaivePersistence**, and under repeated validation the Trends gain remains **positive but modest** and less stable across time.

| Evaluation stage | Baseline | Baseline + Trends / best learned model | Main takeaway |
|---|---:|---:|---|
| First-pass holdout | RMSE ≈ 5.87 | RMSE ≈ 4.80 | Trends improved the learned model |
| Hotel-normalized robustness | RMSE ≈ 6.19 | RMSE ≈ 5.67 | Trends gain survived hotel-wise normalization |
| Fair same-window validation | RMSE = 4.974 | RMSE = 4.798 | Trends gain remained under cleaner comparison |
| Walk-forward validation | RMSE = 8.166 | RMSE = 8.035 | Trends gain was positive but modest |
| Naive benchmark comparison | NaivePersistence RMSE = 4.068 | Best learned RMSE = 4.798 | Persistence remained strongest |

So the ML stage is aligned with the EDA logic, but the final modeling claim must stay limited: Google Trends is not a strong same-day demand proxy; selected lagged search signals can act as an **early supporting indicator**; they improve the learned models somewhat; but they do not yet produce a forecasting system stronger than simple persistence.

## Conclusion

The machine learning stage shows that lagged Google Trends features contain some useful predictive information for daily hotel occupancy. In the learned-model comparisons, adding selected lagged Trends variables improved performance over the baseline model that used only hotel identity, seasonality, and past occupancy. This improvement also remained visible after hotel-wise normalization, fair same-window validation, and walk-forward validation.

However, the final conclusion must remain cautious. The best learned models still did not outperform the strongest simple benchmark, **NaivePersistence**, which predicts occupancy using the previous day's value for the same hotel. This means the forecasting problem is highly persistence-dominated, and recent occupancy history remains stronger than the current external signal set.

Therefore, Google Trends should be interpreted as a **supporting early indicator of travel intent**, not as a standalone or dominant forecasting signal. The project provides partial support for the idea that lagged search behavior helps explain occupancy patterns, but it does not yet prove that Google Trends can produce a better forecasting system than simple short-term persistence.

## Questions and Answers

### Q1. If NaivePersistence is the best model, does that mean the ML stage failed?
No. This is a meaningful finding rather than a failure. NaivePersistence performs strongly because daily hotel occupancy changes gradually; yesterday's occupancy already contains a large amount of information about today's occupancy. The learned models still show that lagged Google Trends improves prediction compared with the learned baseline, but the final benchmark comparison shows that short-term persistence is the strongest signal in the current setup.

### Q2. Why use Google Trends if it does not beat NaivePersistence?
Google Trends is not treated as a direct replacement for occupancy history. It is tested as an external early signal of travel intent. The results suggest that Trends contains some additional information, but this information is weaker than recent occupancy for short-horizon daily prediction. This is still useful because it helps explain demand patterns and may be more valuable for longer planning horizons where yesterday's occupancy is less dominant.

### Q3. Why are same-day Google Trends relationships weak?
Same-day searches do not necessarily convert into same-day hotel occupancy. Resort hotels in Antalya and Alanya are strongly affected by advance planning, agencies, tour operators, and seasonal travel behavior. Because of this, lagged search activity is more reasonable than same-day search activity as a potential predictor. This is why the project focuses on 7, 14, 21, and 28-day lag relationships rather than only same-day correlations.

### Q4. How was data leakage avoided?
Several safeguards were used. Temporal train-test splitting was used instead of random splitting. Occupancy lags were created within each hotel, so values from one hotel did not leak into another. Google Trends lags were created on a unique date-level table before being merged back into the hotel-date table. In the hotel-normalized robustness model, hotel means and standard deviations were computed from the training period only. The fair same-window comparison also forced baseline and Trends models to be evaluated on the same rows and same future test period.

### Q5. Why use both fair same-window validation and walk-forward validation?
They answer different evaluation questions. Fair same-window validation checks whether the baseline and Trends models are compared on the exact same observations and test dates. Walk-forward validation checks whether the conclusion survives across multiple future periods instead of only one holdout split. Together, they make the comparison more reliable than a single train-test split.

### Q6. Why use hotel-wise normalization?
The project combines two hotels that may have different average occupancy levels and different variability. Hotel-wise normalization checks whether the Trends result is only caused by pooling hotels with different baseline occupancy levels. Since the Trends model still improved over the learned baseline after normalization and back-transformation, the result is less likely to be only a cross-hotel level artifact.

### Q7. Why were Ridge Regression and Random Forest selected?
Ridge Regression provides a regularized linear benchmark, while Random Forest provides a non-linear model that can capture interactions between seasonality, hotel identity, lagged occupancy, and lagged Google Trends. The purpose was not to test every possible algorithm, but to compare a simple interpretable learned model with a stronger non-linear learned model under the same validation logic.

### Q8. What is the final answer to the ML research question?
Lagged Google Trends improves the learned models, but it does not create the best overall forecasting system. The strongest benchmark remains NaivePersistence. Therefore, the final answer is partial support: Google Trends has incremental explanatory and predictive value, but daily hotel occupancy forecasting is still mainly driven by recent occupancy and seasonality.

## Related Files

Main scripts:
- `scripts/modeling_baseline_commented.py`
- `scripts/hotel_normalization_robustness_commented.py`
- `scripts/modeling_fair_comparison_commented.py`
- `scripts/modeling_walk_forward_commented.py`
- `scripts/modeling_naive_benchmarks_commented.py`

Main figure folders used in this report:
- `ML/Figures/`
- `ML/Naive_Benchmark/`

Main output folders:
- `model_outputs/baseline_ml/`
- `model_outputs/hotel_normalization_robustness/`
- `model_outputs/fair_same_window_comparison/`
- `model_outputs/walk_forward_validation/`
- `model_outputs/naive_benchmark_comparison/`

Main report tables:
- `reports/ML_reports/model_comparison.csv`
- `reports/ML_reports/model_comparison_by_hotel.csv`
- `reports/ML_reports/model_comparison_same_window.csv`
- `reports/ML_reports/model_comparison_by_hotel_same_window.csv`
- `reports/ML_reports/walk_forward_fold_results.csv`
- `reports/ML_reports/walk_forward_summary.csv`
- `reports/ML_reports/walk_forward_by_hotel.csv`
- `reports/ML_reports/same_window_with_naive_benchmarks.csv`
- `reports/ML_reports/same_window_with_naive_benchmarks_by_hotel.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_fold_results.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_summary.csv`
- `reports/ML_reports/walk_forward_with_naive_benchmarks_by_hotel.csv`
